In [86]:
import numpy as np
import pandas as pd

import statsmodels.api as sm
from scipy.stats import ks_2samp, mannwhitneyu
from sklearn.metrics import average_precision_score

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score, 
    precision_recall_curve,
    classification_report
)
pd.set_option("display.max_rows", None)
from statsmodels.tools.sm_exceptions import PerfectSeparationError

TARGET = "fraud"

%matplotlib inline

## 전체 KEEP 피처 정리

먼저 전체 피처를 성격별로 분류하면:

**Static (카드/사람 속성)**
- `current_age`, `near_retirement_flag`
- `is_prepaid`, `cb_Discover`, `cb_Visa`
- `years_since_pin_change`, `expire_high`
- `mcc_smoothed_risk`, `mcc_risk_level`
- `has_error`, `err_bad_cvv`, `error_count`

**Temporal (시간)**
- `hour_sin`, `hour_cos`, `is_risky_hour`, `weekday`, `is_risky_wd_hour`
- `month_sin`, `is_risky_month`

**Transaction (거래 단위)**
- `log_abs_amount`, `amount_income_ratio`, `amount_limit_ratio`, `limit_ratio_extreme`, `is_refund`

**History (이력 기반)**
- `client_fraud_last1/3`, `card_fraud_last1/3`
- `client_error_last3`, `card_error_last3`
- `log_interval_dev`, `seconds_since_prev_tx`
- `client_mcc_is_new`, `card_mcc_is_new`
- `client_mcc_repeat_cnt_last5`, `card_mcc_repeat_cnt_last5`
- `card_mcc_change_cnt_last5`
- `client_merchant_is_new`, `card_merchant_is_new`
- `merchant_is_new_x_mcc_is_new`, `merchant_change_cnt_last5`
- `velocity_spike_ratio`, `card_velocity_spike_ratio`, `card_tx_1h`
- `vel_x_merchant_new`, `vel_x_card_mcc_new`, `vel_x_client_mcc_new`
- `vel_x_mcc_risk`

---

## Interaction 후보 추천

### Tier 1 — 검증 필수 (이론 강 + 미검증)

이미 EDA_HISTORY에서 velocity×novelty 조합이 강력함을 확인했는데, 아직 건드리지 않은 축들이 있습니다.

**시간 × 행동 패턴** — fraud는 "언제" + "뭘 했냐"의 조합에서 폭발함

| 조합 | 설명 | 기대 이유 |
|---|---|---|
| `is_risky_hour` × `client_mcc_is_new` | 위험 시간대 + MCC 처음 방문 | 심야에 처음 가는 업종 = 매우 의심스러운 패턴 |
| `is_risky_hour` × `client_merchant_is_new` | 위험 시간대 + 가맹점 처음 방문 | 위와 같은 맥락, 가맹점 단위 |
| `is_risky_wd_hour` × `client_mcc_is_new` | 고위험 요일·시간 + 업종 첫 방문 | 이미 검증된 두 신호의 교차 |
| `is_risky_month` × `client_merchant_is_new` | 성수기 + 신규 가맹점 | 성수기 card-not-present 사기 패턴 |

**금액 이탈 × 행동 이상** — baseline에 금액 피처가 있지만, "처음 가는 곳에서 큰 돈" 조합은 미검증

| 조합 | 설명 | 기대 이유 |
|---|---|---|
| `limit_ratio_extreme` × `client_mcc_is_new` | 한도 극단 초과 + 업종 첫 거래 | 한도 초과하면서 처음 가는 업종이면 사기 확률 극대 |
| `limit_ratio_extreme` × `client_merchant_is_new` | 한도 극단 초과 + 가맹점 첫 방문 | 위와 같은 맥락 |
| `amount_limit_ratio` × `is_risky_hour` | 고비율 금액 + 위험 시간대 | 심야의 고비율 거래 |
| `is_refund` × `client_mcc_is_new` | 환불 + 처음 방문 업종 | 처음 가는 업종에서 환불 = 자금 이동 가능성 |

**카드 속성 × 행동 이상** — static 피처 단독은 약하나 행동 이상과 결합 시 강해질 수 있음

| 조합 | 설명 | 기대 이유 |
|---|---|---|
| `is_prepaid` × `client_mcc_is_new` | 선불카드 + 업종 첫 거래 | 선불카드로 처음 가는 업종 = 높은 위험 |
| `is_prepaid` × `velocity_spike_ratio` | 선불카드 + 속도 스파이크 | 선불카드 충전 후 빠른 소진 패턴 |
| `is_prepaid` × `client_merchant_is_new` | 선불카드 + 신규 가맹점 | 선불카드로 처음 가는 가맹점 |
| `cb_Discover` × `client_mcc_is_new` | 고위험 브랜드 + 업종 첫 거래 | 리스크 축 2개 동시 충족 |
| `years_since_pin_change` × `client_merchant_is_new` | PIN 오래 안 바꿈 + 신규 가맹점 | 보안 취약 상태에서 이상 행동 |

---

### Tier 2 — 검증 권장 (보조 신호 강화 가능)

**오류 × 행동 이상**

| 조합 | 설명 |
|---|---|
| `err_bad_cvv` × `client_merchant_is_new` | CVV 오류 + 신규 가맹점 |
| `err_bad_cvv` × `client_mcc_is_new` | CVV 오류 + 업종 첫 거래 |
| `client_error_last3` × `client_merchant_is_new` | 최근 에러 이력 + 신규 가맹점 방문 |
| `has_error` × `is_risky_hour` | 에러 발생 + 위험 시간대 |

**간격 × 행동 이상**

| 조합 | 설명 |
|---|---|
| `log_interval_dev` × `client_mcc_is_new` | 평소보다 짧은 간격 + 처음 업종 |
| `log_interval_dev` × `client_merchant_is_new` | 평소보다 짧은 간격 + 신규 가맹점 |
| `log_interval_dev` × `is_risky_hour` | 짧은 간격 + 위험 시간대 |

**MCC 리스크 × 카드 속성**

| 조합 | 설명 |
|---|---|
| `mcc_smoothed_risk` × `is_prepaid` | 고위험 업종에서 선불카드 사용 |
| `mcc_smoothed_risk` × `near_retirement_flag` | 고위험 업종 + 은퇴 임박 연령 |
| `mcc_risk_level` × `is_risky_hour` | 고위험 업종 + 위험 시간 |

---

### Tier 3 — 선택적 탐색 (약한 신호지만 가능성)

| 조합 | 설명 |
|---|---|
| `near_retirement_flag` × `client_mcc_is_new` | 고령 + 처음 업종 (노인 타겟 사기 패턴) |
| `near_retirement_flag` × `velocity_spike_ratio` | 고령 + 속도 스파이크 |
| `expire_high` × `client_mcc_is_new` | 만료 긴 카드 + 업종 첫 거래 |
| `card_mcc_change_cnt_last5` × `is_risky_hour` | 업종 전환 빈번 + 위험 시간 |
| `merchant_change_cnt_last5` × `amount_limit_ratio` | 가맹점 전환 빈번 + 고비율 금액 |

---

## 우선순위 요약

가장 먼저 검증해야 할 5개를 꼽으면:

1. **`is_risky_hour` × `client_mcc_is_new`** — 시간 이상 × 업종 이상, 두 강한 신호의 미검증 교차
2. **`limit_ratio_extreme` × `client_merchant_is_new`** — 극단 금액 × 신규 가맹점
3. **`is_prepaid` × `velocity_spike_ratio`** — 카드 속성 × 속도 조합 (선불카드는 단독 신호인데 속도와는 미검증)
4. **`err_bad_cvv` × `client_merchant_is_new`** — 두 독립적 이벤트 신호의 동시 발생
5. **`log_interval_dev` × `client_mcc_is_new`** — 간격 이탈 × 업종 신규 (velocity 계열에서 검증됐지만 interval 버전은 미검증)



In [87]:
df = pd.read_parquet("../5DATA/dataset/train_stage2")

In [88]:
BASELINE = [
    # 거래 강도
    "log_abs_amount",
    "amount_income_ratio",
    "amount_limit_ratio",

    # 에러
    "has_error",

    # 고객 맥락 최소
    "credit_limit",
    "current_age",

    # 카드 특성
    "is_credit",
    "has_chip",
]

In [89]:
STATIC_FEATURES = [
    "current_age",
    "near_retirement_flag",
    "is_prepaid",
    "cb_Discover",
    "cb_Visa",
    "years_since_pin_change",
    "expire_high",
    "mcc_smoothed_risk",
    "mcc_risk_level",
    "has_error",
    "err_bad_cvv",
    "error_count",
]

In [90]:
TEMPORAL_FEATURES = [
    "hour_sin",
    "hour_cos",
    "is_risky_hour",
    "weekday",
    "is_risky_wd_hour",
    "month_sin",
    "is_risky_month",
]

In [91]:
TRANSACTION_FEATURES = [
    "log_abs_amount",
    "amount_income_ratio",
    "amount_limit_ratio",
    "limit_ratio_extreme",
    "is_refund",
]

In [92]:
HISTORY_FEATURES = [
    "client_fraud_last1",
    "client_fraud_last3",
    "card_fraud_last1",
    "card_fraud_last3",
    "client_error_last3",
    "card_error_last3",
    "log_interval_dev",
    "seconds_since_prev_tx",
    "client_mcc_is_new",
    "card_mcc_is_new",
    "client_mcc_repeat_cnt_last5",
    "card_mcc_repeat_cnt_last5",
    "card_mcc_change_cnt_last5",
    "client_merchant_is_new",
    "card_merchant_is_new",
    "merchant_is_new_x_mcc_is_new",
    "merchant_change_cnt_last5",
    "velocity_spike_ratio",
    "card_velocity_spike_ratio",
    "card_tx_1h",
    "vel_x_merchant_new",
    "vel_x_card_mcc_new",
    "vel_x_client_mcc_new",
    "vel_x_mcc_risk",
]

In [93]:
ALL = ["id", "fraud"] + BASELINE + STATIC_FEATURES + TEMPORAL_FEATURES + TRANSACTION_FEATURES + HISTORY_FEATURES

In [94]:
DC  = df.columns

In [95]:
INTER = list(set(ALL) & set(DC))
INTER

['is_prepaid',
 'is_refund',
 'amount_limit_ratio',
 'err_bad_cvv',
 'years_since_pin_change',
 'cb_Visa',
 'is_credit',
 'amount_income_ratio',
 'cb_Discover',
 'current_age',
 'credit_limit',
 'log_abs_amount',
 'has_error',
 'id',
 'has_chip',
 'fraud',
 'weekday']

# Make Dataset for INTERACTION

## Static

In [96]:
def retirement_scan(df):
    base = df["fraud"].mean()
    s = df["years_to_retirement"]

    results = []

    for t in [5, 8, 10, 12, 15]:
        flag = (s <= t).astype(int)
        support = flag.mean()
        rate = df.loc[flag==1, "fraud"].mean()
        lift = rate / base

        results.append((t, support, rate, lift))

    return pd.DataFrame(results, columns=["threshold","support","fraud_rate","lift"])

retirement_scan(df)

df["near_retirement_flag"] = (df["years_to_retirement"] <= 8).astype("int8")

In [97]:
df["expire_high"] = (df["months_to_expire"] >= 146).astype(int)

In [98]:
MCC_COL = "mcc"
LABEL = "fraud"
alpha = 1000

# 1. Global fraud rate
global_rate = df[LABEL].mean()

# 2. MCC별 거래수 + raw fraud rate 계산
mcc_stats = (
    df.groupby(MCC_COL)[LABEL]
      .agg(tx_count="count", raw_rate="mean")
      .reset_index()
)

# 3. Bayesian smoothing 적용
mcc_stats["mcc_smoothed_risk"] = (
    (mcc_stats["raw_rate"] * mcc_stats["tx_count"] +
     global_rate * alpha)
    / (mcc_stats["tx_count"] + alpha)
)

# 4. 원본 df에 매핑
df["mcc_smoothed_risk"] = df[MCC_COL].map(
    mcc_stats.set_index(MCC_COL)["mcc_smoothed_risk"]
)

df["mcc_smoothed_risk"] = df["mcc_smoothed_risk"].astype("float32")

In [99]:
# 상위 10%
q = 0.9

threshold = df["mcc_smoothed_risk"].quantile(q)

df["mcc_is_highrisk"] = (
    df["mcc_smoothed_risk"] >= threshold
).astype("int8")

print("threshold:", threshold)

threshold: 0.01955270953476429


In [100]:
high = ["5732", "5712", "5045", "5816", "5651", "4411"]
mid  = ["5193", "5311", "7996"]

df["mcc_risk_level"] = 0
df.loc[df["mcc"].isin(mid), "mcc_risk_level"] = 1
df.loc[df["mcc"].isin(high), "mcc_risk_level"] = 2

In [101]:
df = df.sort_values(['card_id', 'date'])

df['card_fraud_cum_prev'] = (
    df.groupby('card_id')['fraud']
      .cumsum()
      .shift(1)
      .fillna(0)
)
err_type_cols = [
    'err_bad_cvv',
    'err_bad_card_number',
    'err_bad_expiration',
    'err_insufficient_balance',
    'err_technical_glitch'
]
df['error_count'] = df[err_type_cols].sum(axis=1)

## Temporal

In [102]:
df["hour_sin"] = np.sin(2 * np.pi * df["tx_hour"] / 24).astype("float32")
df["hour_cos"] = np.cos(2 * np.pi * df["tx_hour"] / 24).astype("float32")

In [103]:
RISKY_HOURS = [3, 5]
df['is_risky_hour'] = df['tx_hour'].isin(RISKY_HOURS).astype('int8')
df.groupby('is_risky_hour')['fraud'].mean() 

is_risky_hour
0    0.010579
1    0.019665
Name: fraud, dtype: float64

In [104]:
base_rate = df["fraud"].mean()

pivot_rate = df.pivot_table(
    index="weekday", columns="tx_hour", values="fraud", aggfunc="mean"
)

max_idx = pivot_rate.stack().idxmax()
max_rate = float(pivot_rate.loc[max_idx[0], max_idx[1]])
max_lift = max_rate / base_rate

max_idx, max_rate, max_lift

TOPN = 10
top_cells = pivot_rate.stack().sort_values(ascending=False).head(TOPN)
top_pairs = list(top_cells.index)  # [(weekday, hour), ...]

top_pairs[:10], top_cells.head(10)

top_pair_set = set(top_pairs)
df["is_risky_wd_hour"] = [
    int((wd, hr) in top_pair_set)
    for wd, hr in zip(df["weekday"], df["tx_hour"])
]
df["is_risky_wd_hour"] = df["is_risky_wd_hour"].astype("int8")

In [105]:
df["month_sin"] = np.sin(2 * np.pi * df["tx_month"] / 12).astype("float32")

In [106]:
HIGH_RISK_MONTHS = [7,8,10,12]
df["is_risky_month"] = df["tx_month"].isin(HIGH_RISK_MONTHS).astype("int8")

## Transaction

In [107]:
thr = df["amount_limit_ratio"].quantile(0.999)
df[df["amount_limit_ratio"] >= thr]["fraud"].mean()
df["limit_ratio_extreme"] = (df["amount_limit_ratio"] >= thr).astype(int)

## History

In [108]:
df["client_fraud_last1"] = (
    df.groupby("client_id")["fraud"]
      .shift(1)
      .fillna(0)
      .astype("int8")
)
df["card_fraud_last1"] = (
    df.groupby("card_id")["fraud"]
      .shift(1)
      .fillna(0)
      .astype("int8")
)
f1 = df.groupby("client_id")["fraud"].shift(1)
f2 = df.groupby("client_id")["fraud"].shift(2)
f3 = df.groupby("client_id")["fraud"].shift(3)

df["client_fraud_last3"] = (
    f1.fillna(0).astype("int8") +
    f2.fillna(0).astype("int8") +
    f3.fillna(0).astype("int8")
)
f1 = df.groupby("card_id")["fraud"].shift(1)
f2 = df.groupby("card_id")["fraud"].shift(2)
f3 = df.groupby("card_id")["fraud"].shift(3)

df["card_fraud_last3"] = (
    f1.fillna(0).astype("int8") +
    f2.fillna(0).astype("int8") +
    f3.fillna(0).astype("int8")
)

In [109]:
g = df.groupby("client_id")["has_error"]

e1 = g.shift(1).fillna(0).astype("int8")
e2 = g.shift(2).fillna(0).astype("int8")
e3 = g.shift(3).fillna(0).astype("int8")
df["client_error_last3"] = (e1 + e2 + e3).astype("int8")

g = df.groupby("card_id")["has_error"]

e1 = g.shift(1).fillna(0).astype("int8")
e2 = g.shift(2).fillna(0).astype("int8")
e3 = g.shift(3).fillna(0).astype("int8")
df["card_error_last3"] = (e1 + e2 + e3).astype("int8")


In [110]:
# 1) 이전 거래 시점
df["prev_tx_time"] = df.groupby("client_id")["date"].shift(1)

# 2) 초 단위 간격
df["seconds_since_prev_tx"] = (
    (df["date"] - df["prev_tx_time"]).dt.total_seconds()
)

# 첫 거래는 간격 없음 → 큰 값으로 처리 (중립)
df["seconds_since_prev_tx"] = df["seconds_since_prev_tx"].fillna(0)

# 로그 변환
df["log_interval"] = np.log1p(df["seconds_since_prev_tx"])

/home/nakyung/.local/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [111]:
# 과거값 shift
df["log_interval_shift"] = df.groupby("client_id")["log_interval"].shift(1)

# 누적합
df["interval_cumsum"] = (
    df["log_interval_shift"].fillna(0)
      .groupby(df["client_id"])
      .cumsum()
)

# 과거 개수
df["interval_cnt_past"] = df.groupby("client_id").cumcount()

# 과거 평균
df["client_avg_interval_prev"] = np.where(
    df["interval_cnt_past"] > 0,
    df["interval_cumsum"] / df["interval_cnt_past"],
    df["log_interval"] 
)


In [112]:
# ratio
df["interval_ratio"] = (
    df["log_interval"] /
    (df["client_avg_interval_prev"] + 1e-6)
)

# deviation (z-score 방식)
df["log_interval_dev"] = (
    df["log_interval"] -
    df["client_avg_interval_prev"]
)

In [113]:
df["client_mcc_prior_count"] = df.groupby(["client_id", "mcc"]).cumcount()
df["client_mcc_is_new"] = (df["client_mcc_prior_count"] == 0).astype("int8")

In [114]:
df["card_mcc_prior_count"] = df.groupby(["card_id", "mcc"]).cumcount()
df["card_mcc_is_new"] = (df["card_mcc_prior_count"] == 0).astype("int8")

In [115]:
df = df.sort_values(["client_id", "date"]).copy()
df["mcc"] = df["mcc"].astype("category")
g = df.groupby("client_id")["mcc"]
m1 = (df["mcc"] == g.shift(1))
m2 = (df["mcc"] == g.shift(2))
m3 = (df["mcc"] == g.shift(3))
m4 = (df["mcc"] == g.shift(4))
m5 = (df["mcc"] == g.shift(5))
df["client_mcc_repeat_cnt_last5"] = (m1.fillna(False).astype("int8")
                                   + m2.fillna(False).astype("int8")
                                   + m3.fillna(False).astype("int8")
                                   + m4.fillna(False).astype("int8")
                                   + m5.fillna(False).astype("int8"))

In [116]:
g = df.groupby("card_id")["mcc"]
m1 = (df["mcc"] == g.shift(1))
m2 = (df["mcc"] == g.shift(2))
m3 = (df["mcc"] == g.shift(3))
m4 = (df["mcc"] == g.shift(4))
m5 = (df["mcc"] == g.shift(5))
df["card_mcc_repeat_cnt_last5"] = (m1.fillna(False).astype("int8")
                                   + m2.fillna(False).astype("int8")
                                   + m3.fillna(False).astype("int8")
                                   + m4.fillna(False).astype("int8")
                                   + m5.fillna(False).astype("int8"))

In [117]:
prev = g.shift(1)
prev2 = g.shift(2)
prev3 = g.shift(3)
prev4 = g.shift(4)
prev5 = g.shift(5)

chg1 = (prev  != prev2)
chg2 = (prev2 != prev3)
chg3 = (prev3 != prev4)
chg4 = (prev4 != prev5)

df["card_mcc_change_cnt_last5"] = (chg1.fillna(False).astype("int8")
                                  + chg2.fillna(False).astype("int8")
                                  + chg3.fillna(False).astype("int8")
                                  + chg4.fillna(False).astype("int8"))


In [118]:
df["client_merchant_is_new"] = (
    df.groupby(["client_id", "merchant_id"], sort=False).cumcount().eq(0).astype("int8")
)

df["card_merchant_is_new"] = (
    df.groupby(["card_id", "merchant_id"], sort=False).cumcount().eq(0).astype("int8")
)

In [119]:
# merchant_is_new 정의(카드 기준)
df["merchant_is_new"] = df["card_merchant_is_new"].astype("int8")

# merchant_is_new × mcc_is_new 
if "card_mcc_is_new" in df.columns:
    df["merchant_is_new_x_mcc_is_new"] = (
        df["merchant_is_new"].astype("int8") * df["card_mcc_is_new"].astype("int8")
    ).astype("int8")
else:
    # 없으면 대체: card 단위로 mcc 첫 등장 여부 생성
    df["card_mcc_is_new"] = (
        df.groupby(["card_id", "mcc"], sort=False).cumcount().eq(0).astype("int8")
    )
    df["merchant_is_new_x_mcc_is_new"] = (
        df["merchant_is_new"] * df["card_mcc_is_new"]
    ).astype("int8")

In [120]:
prev_merchant = df.groupby("card_id", sort=False)["merchant_id"].shift(1)

df["merchant_changed"] = (
    df["merchant_id"].ne(prev_merchant)      # 직전과 다르면 True
    .fillna(True)                            # 첫 거래는 변경
    .astype("int8")
)

# 2) 최근 5건에서 변경 횟수
df["merchant_change_cnt_last5"] = (
    df.groupby("card_id", sort=False)["merchant_changed"]
      .rolling(window=5, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
      .astype("int8")
)

df.drop(columns=["merchant_changed"], inplace=True)

In [121]:
df = df.sort_values(["client_id", "date"]).reset_index(drop=True)

# numpy index 준비
n = len(df)

client_tx_1h = np.zeros(n, dtype=np.int32)
card_tx_1h = np.zeros(n, dtype=np.int32)

for cid, idx in df.groupby("client_id").groups.items():
    g = df.loc[idx]
    times = g["date"].values.astype("datetime64[s]").astype("int64")
    
    # 누적 거래 번호
    cum = np.arange(len(g))
    
    # 1시간 전 timestamp
    t_minus_1h = times - 3600
    
    left = np.searchsorted(times, t_minus_1h)
    
    client_tx_1h[idx] = cum - left + 1

df["client_tx_1h"] = client_tx_1h

df["client_tx_1h_shift"] = df.groupby("client_id")["client_tx_1h"].shift(1)

# 지금까지의 (과거) 1시간 거래 수 총합
df["client_tx_1h_cumsum"] = (
    df["client_tx_1h_shift"].fillna(0)
      .groupby(df["client_id"])
      .cumsum()
)

# 과거 거래 개수
df["client_tx_cnt_past"] = df.groupby("client_id").cumcount()

# 과거 평균 계산
df["client_tx_1h_avg_prev"] = np.where(
    df["client_tx_cnt_past"] > 0,
    df["client_tx_1h_cumsum"] / df["client_tx_cnt_past"],
    df["client_tx_1h"]
)

# 현재 1시간 거래 수가 평소 평균 대비 몇 배인가? 
df["velocity_spike_ratio"] = (
    df["client_tx_1h"] /
    (df["client_tx_1h_avg_prev"] + 1e-6)
)

for cid, idx in df.groupby("card_id").groups.items():
    g = df.loc[idx]
    times = g["date"].values.astype("datetime64[s]").astype("int64")
    
    cum = np.arange(len(g))
    t_minus_1h = times - 3600
    
    left = np.searchsorted(times, t_minus_1h)
    
    card_tx_1h[idx] = cum - left + 1

df["card_tx_1h"] = card_tx_1h

df["card_tx_1h_shift"] = df.groupby("card_id")["card_tx_1h"].shift(1)

df["card_tx_1h_cumsum"] = (
    df["card_tx_1h_shift"].fillna(0)
      .groupby(df["card_id"])
      .cumsum()
)

df["card_tx_cnt_past"] = df.groupby("card_id").cumcount()

df["card_tx_1h_avg_prev"] = np.where(
    df["card_tx_cnt_past"] > 0,
    df["card_tx_1h_cumsum"] / df["card_tx_cnt_past"],
    df["card_tx_1h"]
)

df["card_velocity_spike_ratio"] = (
    df["card_tx_1h"] /
    (df["card_tx_1h_avg_prev"] + 1e-6)
)

In [122]:
df["vel_x_merchant_new"] = (
    df["velocity_spike_ratio"] *
    df["merchant_is_new"]
)
df["vel_x_card_mcc_new"] = (
    df["velocity_spike_ratio"] *
    df["card_mcc_is_new"]
)
df["vel_x_client_mcc_new"] = (
    df["velocity_spike_ratio"] *
    df["client_mcc_is_new"]
)
df["vel_x_mcc_risk"] = (
    df["velocity_spike_ratio"] *
    df["mcc_risk_level"]
)

---

### **Tier 1**

시간 x 행동 패턴

In [123]:
# 1) 위험 시간대 × 업종 첫 방문
df["risky_hour_x_client_mcc_new"] = (
    df["is_risky_hour"] * df["client_mcc_is_new"]
)

# 2) 위험 시간대 × 가맹점 첫 방문
df["risky_hour_x_client_merchant_new"] = (
    df["is_risky_hour"] * df["client_merchant_is_new"]
)

# 3) 고위험 요일·시간 × 업종 첫 방문
df["risky_wdhour_x_client_mcc_new"] = (
    df["is_risky_wd_hour"] * df["client_mcc_is_new"]
)

# 4) 성수기 × 신규 가맹점
df["risky_month_x_client_merchant_new"] = (
    df["is_risky_month"] * df["client_merchant_is_new"]
)

금액 이탈 x 행동 이상

In [124]:
# 1) 한도 극단 초과 × 업종 첫 거래
df["limit_extreme_x_client_mcc_new"] = (
    df["limit_ratio_extreme"] * df["client_mcc_is_new"]
)

# 2) 한도 극단 초과 × 가맹점 첫 방문
df["limit_extreme_x_client_merchant_new"] = (
    df["limit_ratio_extreme"] * df["client_merchant_is_new"]
)

# 3) 고비율 금액 × 위험 시간대
df["amount_limit_ratio_x_risky_hour"] = (
    df["amount_limit_ratio"] * df["is_risky_hour"]
)

# 4) 환불 × 처음 방문 업종
df["refund_x_client_mcc_new"] = (
    df["is_refund"] * df["client_mcc_is_new"]
)

카드 속성 × 행동 이상 

In [125]:
# 1) 선불카드 × 업종 첫 거래
df["is_prepaid_x_client_mcc_new"] = (
    df["is_prepaid"] * df["client_mcc_is_new"]
)

# 2) 선불카드 × 속도 스파이크
df["is_prepaid_x_velocity_spike"] = (
    df["is_prepaid"] * df["velocity_spike_ratio"]
)

# 3) 선불카드 × 신규 가맹점
df["is_prepaid_x_client_merchant_new"] = (
    df["is_prepaid"] * df["client_merchant_is_new"]
)

# 4) Discover 브랜드 × 업종 첫 거래
df["cb_Discover_x_client_mcc_new"] = (
    df["cb_Discover"] * df["client_mcc_is_new"]
)

# 5) PIN 오래 미변경 × 신규 가맹점
df["years_since_pin_change_x_client_merchant_new"] = (
    df["years_since_pin_change"] * df["client_merchant_is_new"]
)

### **Tier2**

오류 × 행동 이상

In [126]:
# 1) CVV 오류 × 신규 가맹점
df["err_bad_cvv_x_client_merchant_new"] = (
    df["err_bad_cvv"] * df["client_merchant_is_new"]
)

# 2) CVV 오류 × 업종 첫 거래
df["err_bad_cvv_x_client_mcc_new"] = (
    df["err_bad_cvv"] * df["client_mcc_is_new"]
)

# 3) 최근 에러 이력 × 신규 가맹점 방문
df["client_error_last3_x_client_merchant_new"] = (
    df["client_error_last3"] * df["client_merchant_is_new"]
)

# 4) 에러 발생 × 위험 시간대
df["has_error_x_risky_hour"] = (
    df["has_error"] * df["is_risky_hour"]
)

간격 × 행동 이상

In [127]:
# 1) 평소보다 짧은 간격 × 처음 업종
df["log_interval_dev_x_client_mcc_new"] = (
    df["log_interval_dev"] * df["client_mcc_is_new"]
)

# 2) 평소보다 짧은 간격 × 신규 가맹점
df["log_interval_dev_x_client_merchant_new"] = (
    df["log_interval_dev"] * df["client_merchant_is_new"]
)

# 3) 평소보다 짧은 간격 × 위험 시간대
df["log_interval_dev_x_risky_hour"] = (
    df["log_interval_dev"] * df["is_risky_hour"]
)

MCC 리스크 × 카드 속성

In [128]:
# 1) 고위험 업종에서 선불카드 사용
df["mcc_smoothed_risk_x_is_prepaid"] = (
    df["mcc_smoothed_risk"] * df["is_prepaid"]
)

# 2) 고위험 업종 + 은퇴 임박 연령
df["mcc_smoothed_risk_x_near_retirement"] = (
    df["mcc_smoothed_risk"] * df["near_retirement_flag"]
)

# 3) 고위험 업종 + 위험 시간
df["mcc_risk_level_x_risky_hour"] = (
    df["mcc_risk_level"] * df["is_risky_hour"]
)

### **Tier 3**

In [129]:
# 1) 고령 × 처음 업종
df["near_retirement_x_client_mcc_new"] = (
    df["near_retirement_flag"] * df["client_mcc_is_new"]
)

# 2) 고령 × 속도 스파이크
df["near_retirement_x_velocity_spike"] = (
    df["near_retirement_flag"] * df["velocity_spike_ratio"]
)

# 3) 만료 긴 카드 × 업종 첫 거래
df["expire_high_x_client_mcc_new"] = (
    df["expire_high"] * df["client_mcc_is_new"]
)

# 4) 업종 전환 빈번 × 위험 시간
df["card_mcc_change_last5_x_risky_hour"] = (
    df["card_mcc_change_cnt_last5"] * df["is_risky_hour"]
)

# 5) 가맹점 전환 빈번 × 고비율 금액
df["merchant_change_last5_x_amount_limit_ratio"] = (
    df["merchant_change_cnt_last5"] * df["amount_limit_ratio"]
)

In [130]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import average_precision_score

LABEL = "fraud"

INTER_COLS = [
    col for col in df.columns
    if "_x_" in col
]

In [131]:
def evaluate_feature(df, col, label=LABEL):

    tmp = df[[col, label]].dropna().copy()

    if tmp[col].nunique() <= 1:
        return None

    X = sm.add_constant(tmp[[col]])
    y = tmp[label].astype(int)

    try:
        model = sm.Logit(y, X).fit(disp=0)
    except:
        return None

    coef = model.params[col]
    pval = model.pvalues[col]
    OR = np.exp(coef)

    # score
    score = model.predict(X)

    # Top decile lift
    top_n = int(len(score) * 0.1)
    idx = np.argsort(-score)[:top_n]

    base_rate = y.mean()
    top_rate = y.iloc[idx].mean()
    lift = top_rate / base_rate if base_rate > 0 else np.nan

    return {
        "feature": col,
        "coef": coef,
        "OR": OR,
        "p_value": pval,
        "top_decile_rate": top_rate,
        "top_decile_lift": lift,
        "base_rate": base_rate,
        "n": len(tmp)
    }

In [132]:
results = []

for col in INTER_COLS:
    r = evaluate_feature(df, col)
    if r is not None:
        results.append(r)

results_df = (
    pd.DataFrame(results)
      .sort_values("p_value")
)

results_df

,feature,coef,OR,p_value,top_decile_rate,top_decile_lift,base_rate,n
0,merchant_is_new_x_mcc_is_new,3.427844,3.081015e+01,0.000000e+00,0.069803,6.466200,0.010795,608430
1,vel_x_merchant_new,3.150668,2.335166e+01,0.000000e+00,0.077741,7.201583,0.010795,608430
2,vel_x_card_mcc_new,3.128710,2.284449e+01,0.000000e+00,0.069819,6.467722,0.010795,608430
3,vel_x_client_mcc_new,3.745918,4.234788e+01,0.000000e+00,0.064560,5.980512,0.010795,608430
4,vel_x_mcc_risk,1.298903,3.665273e+00,0.000000e+00,0.034466,3.192753,0.010795,608430
7,risky_wdhour_x_client_mcc_new,4.238091,6.927545e+01,0.000000e+00,0.020282,1.878806,0.010795,608430
13,is_prepaid_x_client_mcc_new,4.192401,6.618153e+01,0.000000e+00,0.017800,1.648904,0.010795,608430
8,risky_month_x_client_merchant_new,3.300402,2.712354e+01,0.000000e+00,0.040087,3.713459,0.010795,608430
15,is_prepaid_x_client_merchant_new,3.949300,5.189903e+01,0.000000e+00,0.019904,1.843788,0.010795,608430
26,mcc_smoothed_risk_x_near_retirement,50.579517,9.255611e+21,0.000000e+00,0.044722,4.142814,0.010795,608430


In [133]:
SIGNIFICANT = results_df[
    (results_df["p_value"] < 0.05) &
    (results_df["OR"] > 1.05)
]

SIGNIFICANT.sort_values("top_decile_lift", ascending=False)

,feature,coef,OR,p_value,top_decile_rate,top_decile_lift,base_rate,n
1,vel_x_merchant_new,3.150668,2.335166e+01,0.000000e+00,0.077741,7.201583,0.010795,608430
2,vel_x_card_mcc_new,3.128710,2.284449e+01,0.000000e+00,0.069819,6.467722,0.010795,608430
0,merchant_is_new_x_mcc_is_new,3.427844,3.081015e+01,0.000000e+00,0.069803,6.466200,0.010795,608430
3,vel_x_client_mcc_new,3.745918,4.234788e+01,0.000000e+00,0.064560,5.980512,0.010795,608430
17,years_since_pin_change_x_client_merchant_new,0.722785,2.060162e+00,0.000000e+00,0.057196,5.298417,0.010795,608430
26,mcc_smoothed_risk_x_near_retirement,50.579517,9.255611e+21,0.000000e+00,0.044722,4.142814,0.010795,608430
8,risky_month_x_client_merchant_new,3.300402,2.712354e+01,0.000000e+00,0.040087,3.713459,0.010795,608430
4,vel_x_mcc_risk,1.298903,3.665273e+00,0.000000e+00,0.034466,3.192753,0.010795,608430
32,merchant_change_last5_x_amount_limit_ratio,0.255191,1.290709e+00,2.990522e-268,0.033069,3.063337,0.010795,608430
28,near_retirement_x_client_mcc_new,3.641562,3.815138e+01,0.000000e+00,0.032773,3.035932,0.010795,608430


다변량 logit

In [134]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

LABEL = "fraud"

sig_cols = SIGNIFICANT["feature"].tolist()

# 1) X, y 구성
tmp = df[sig_cols + [LABEL]].copy()
tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna()

y = tmp[LABEL].astype(int)
X = tmp[sig_cols]

# 2) 상수항 추가
X = sm.add_constant(X, has_constant="add")

# 3) 다변량 Logit 적합 (기본)
multi_model = sm.Logit(y, X).fit(disp=0)
print(multi_model.summary())

# 4) 결과 표 (coef, OR, p-value)
multi_res = pd.DataFrame({
    "feature": multi_model.params.index,
    "coef": multi_model.params.values,
    "OR": np.exp(multi_model.params.values),
    "p_value": multi_model.pvalues.values,
}).sort_values("p_value")

multi_res

                           Logit Regression Results                           
Dep. Variable:                  fraud   No. Observations:               608430
Model:                          Logit   Df Residuals:                   608399
Method:                           MLE   Df Model:                           30
Date:                Tue, 24 Feb 2026   Pseudo R-squ.:                  0.4902
Time:                        10:42:01   Log-Likelihood:                -18495.
converged:                       True   LL-Null:                       -36277.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
const                                           -6.3242      0.033   -189.494      0.000      -6.390      -6.259
merchant_is_new_x_mcc_is_new 

,feature,coef,OR,p_value
0,const,-6.324222,1.792360e-03,0.000000e+00
2,vel_x_merchant_new,2.152840,8.609277e+00,0.000000e+00
10,mcc_smoothed_risk_x_near_retirement,32.568446,1.394107e+14,0.000000e+00
4,vel_x_client_mcc_new,2.129374,8.409598e+00,5.294113e-236
14,years_since_pin_change_x_client_merchant_new,0.242099,1.273920e+00,3.291267e-176
8,risky_month_x_client_merchant_new,0.840838,2.318309e+00,1.924263e-97
17,merchant_change_last5_x_amount_limit_ratio,0.129933,1.138752e+00,1.979424e-62
12,near_retirement_x_client_mcc_new,-1.090215,3.361443e-01,1.179161e-51
15,mcc_smoothed_risk_x_is_prepaid,46.787243,2.086611e+20,5.554491e-42
6,risky_wdhour_x_client_mcc_new,0.998730,2.714831e+00,9.495063e-38


In [135]:
import numpy as np
import pandas as pd

LABEL = "fraud"

def support_table(df, col, label=LABEL):
    s = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    tmp = df.loc[s.index, [col, label]].copy()

    # binary-like support (0 vs >0)
    flag = (tmp[col] > 0).astype(int)

    out = (
        tmp.assign(flag=flag)
           .groupby("flag")[label]
           .agg(["mean", "count"])
           .rename(columns={"mean":"fraud_rate", "count":"n"})
    )
    out["support"] = out["n"] / len(tmp)
    return out

for c in ["mcc_smoothed_risk_x_near_retirement", "mcc_smoothed_risk_x_is_prepaid"]:
    print(c)
    display(support_table(df, c))

mcc_smoothed_risk_x_near_retirement


,fraud_rate,n,support
flag,,,
0,0.009634,393801,0.647241
1,0.012925,214629,0.352759


mcc_smoothed_risk_x_is_prepaid


,fraud_rate,n,support
flag,,,
0,0.010276,562258,0.924113
1,0.017110,46172,0.075887


VIF check

In [136]:
X_no_const = X.drop(columns=["const"], errors="ignore")

vif = pd.DataFrame({
    "feature": X_no_const.columns,
    "VIF": [variance_inflation_factor(X_no_const.values, i) for i in range(X_no_const.shape[1])]
}).sort_values("VIF", ascending=False)

vif

,feature,VIF
2,vel_x_card_mcc_new,25.330814
0,merchant_is_new_x_mcc_is_new,21.135980
17,limit_extreme_x_client_merchant_new,4.025754
21,limit_extreme_x_client_mcc_new,3.948558
3,vel_x_client_mcc_new,3.409520
1,vel_x_merchant_new,2.738545
6,is_prepaid_x_client_mcc_new,1.944172
8,is_prepaid_x_client_merchant_new,1.807414
11,near_retirement_x_client_mcc_new,1.745909
23,err_bad_cvv_x_client_mcc_new,1.639889


In [137]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df["mcc_smoothed_risk_scaled"] = scaler.fit_transform(
    df[["mcc_smoothed_risk"]]
)

df["mcc_smoothed_risk_scaled_x_is_prepaid"] = (
    df["mcc_smoothed_risk_scaled"] * df["is_prepaid"]
)

df["mcc_smoothed_risk_scaled_x_near_retirement"] = (
    df["mcc_smoothed_risk_scaled"] * df["near_retirement_flag"]
)

In [138]:
DROP_COLS = [
    "mcc_smoothed_risk_x_is_prepaid",
    "mcc_smoothed_risk_x_near_retirement",
]

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

In [139]:
NEW_COLS = [
    "mcc_smoothed_risk_scaled_x_is_prepaid",
    "mcc_smoothed_risk_scaled_x_near_retirement",
]

sig_cols_updated = [
    c for c in sig_cols if c not in DROP_COLS
] + NEW_COLS

In [140]:
import statsmodels.api as sm
import numpy as np

tmp = df[sig_cols_updated + ["fraud"]].replace([np.inf, -np.inf], np.nan).dropna()

y = tmp["fraud"].astype(int)
X = sm.add_constant(tmp[sig_cols_updated], has_constant="add")

model = sm.Logit(y, X).fit(disp=0)

multi_res = pd.DataFrame({
    "feature": model.params.index,
    "coef": model.params.values,
    "OR": np.exp(model.params.values),
    "p_value": model.pvalues.values,
}).sort_values("p_value")

multi_res

,feature,coef,OR,p_value
0,const,-6.304631,0.001828,0.000000e+00
2,vel_x_merchant_new,2.137529,8.478458,0.000000e+00
30,mcc_smoothed_risk_scaled_x_near_retirement,0.656985,1.928969,0.000000e+00
4,vel_x_client_mcc_new,2.121498,8.343630,4.378116e-234
13,years_since_pin_change_x_client_merchant_new,0.242509,1.274443,2.924846e-176
8,risky_month_x_client_merchant_new,0.843825,2.325243,7.255285e-98
15,merchant_change_last5_x_amount_limit_ratio,0.130182,1.139036,2.064852e-61
11,near_retirement_x_client_mcc_new,-1.079104,0.339900,1.782764e-50
29,mcc_smoothed_risk_scaled_x_is_prepaid,0.983694,2.674316,1.811996e-42
6,risky_wdhour_x_client_mcc_new,0.999258,2.716266,9.951157e-38


***핵심 interaction***

vel_x_merchant_new

vel_x_client_mcc_new

mcc_smoothed_risk_scaled_x_is_prepaid

mcc_smoothed_risk_scaled_x_near_retirement

years_since_pin_change_x_client_merchant_new

risky_month_x_client_merchant_new

merchant_change_last5_x_amount_limit_ratio

risky_wdhour_x_client_mcc_new

client_error_last3_x_client_merchant_new

err_bad_cvv_x_client_merchant_new

limit_extreme_x_client_merchant_new

is_prepaid_x_client_merchant_new

vel_x_mcc_risk

refund_x_client_mcc_new

---

In [141]:
FINAL_INTERACTIONS = [
    "vel_x_merchant_new",
    "vel_x_client_mcc_new",
    "mcc_smoothed_risk_scaled_x_is_prepaid",
    "mcc_smoothed_risk_scaled_x_near_retirement",
    "years_since_pin_change_x_client_merchant_new",
    "risky_month_x_client_merchant_new",
    "merchant_change_last5_x_amount_limit_ratio",
    "risky_wdhour_x_client_mcc_new",
    "client_error_last3_x_client_merchant_new",
    "err_bad_cvv_x_client_merchant_new",
    "limit_extreme_x_client_merchant_new",
    "is_prepaid_x_client_merchant_new",
    "vel_x_mcc_risk",
    "refund_x_client_mcc_new",
]

In [142]:
FINAL_COLS = ALL + FINAL_INTERACTIONS

In [143]:
FINAL_COLS_EXIST = [c for c in FINAL_COLS if c in df.columns]

df = df[FINAL_COLS_EXIST].copy()

In [144]:
missing_cols = [c for c in FINAL_COLS if c not in df.columns]
missing_cols

[]

In [145]:
FINAL_COLS = list(dict.fromkeys(ALL + FINAL_INTERACTIONS))
df = df[FINAL_COLS].copy()

In [146]:
df.columns.duplicated().sum()

np.int64(8)

In [147]:
df = df.loc[:, ~df.columns.duplicated()].copy()

In [148]:
df.columns.duplicated().sum()

np.int64(0)

In [149]:
df.to_parquet("DATA")